# Actividad 2 — Diagnóstico de diabetes con árboles de decisión
**Aprendizaje Supervisado · Sesión 9 · Naive Bayes y árboles de clasificación**

**Qué haremos.** En la presentación vimos cómo CART elige cada corte minimizando la **impureza de
Gini** o maximizando la **ganancia de información** (entropía), y cómo leer una curva **ROC** y su
**AUC**. Hoy entrenas árboles de clasificación sobre un conjunto sintético de diagnóstico de diabetes
(inspirado en el clásico *Pima Indians Diabetes*) y los comparas contra `GaussianNB`.

Trabaja **de forma individual**. Las celdas con `...` están incompletas: reemplaza los puntos
suspensivos con el código que falta (el comentario `✏️` te da una pista de qué se espera).


## Parte 0 — Los datos

Cada fila es una paciente con 6 variables de entrada (`embarazos`, `glucosa_mgdl`, `presion_sistolica`,
`imc`, `insulina_uUml`, `edad_anios`) y la etiqueta `diabetico` (1/0). El riesgo real depende sobre todo
de la glucosa y el IMC, con un efecto **no lineal**: el riesgo se dispara una vez que la glucosa supera
los ~150 mg/dL, de forma que ni un árbol pequeño ni Naive Bayes lo resuelven perfectamente — como en la
vida real.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                              roc_curve, roc_auc_score)

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def genera_diabetes(n, semilla):
    rng = np.random.default_rng(semilla)
    embarazos = rng.poisson(1.8, n).clip(0, 14).astype(float)
    edad = np.clip(rng.gamma(4.0, 8.0, n) + 18, 18, 80).round(0)
    glucosa = np.clip(rng.normal(118, 28, n), 55, 240)
    presion = np.clip(rng.normal(72, 12, n) + 0.05 * (edad - 40), 40, 120)
    imc = np.clip(rng.normal(31, 6.5, n) + 0.02 * (glucosa - 118), 15, 55)
    insulina = np.clip(rng.lognormal(np.log(90), 0.55, n) + 0.6 * (glucosa - 118), 15, 600)
    z = (-9.6 + 0.040 * glucosa + 0.085 * imc + 0.018 * edad + 0.085 * embarazos
         + 0.004 * insulina - 0.00035 * (glucosa - 150) ** 2)
    p = 1 / (1 + np.exp(-z))
    diabetico = (rng.uniform(0, 1, n) < p).astype(int)
    return pd.DataFrame({"embarazos": embarazos, "glucosa_mgdl": glucosa.round(1),
                          "presion_sistolica": presion.round(1), "imc": imc.round(1),
                          "insulina_uUml": insulina.round(1), "edad_anios": edad, "diabetico": diabetico})


diabetes = genera_diabetes(2400, semilla=2026)
print(diabetes.shape)
diabetes.head()


In [ ]:
prop = diabetes["diabetico"].value_counts(normalize=True)
print((prop * 100).round(1), "%")
diabetes.groupby("diabetico")[["glucosa_mgdl", "imc", "edad_anios"]].mean().round(1)


✏️ **Pregunta 0.1.** (a) ¿Qué proporción de pacientes es diabética? (b) ¿Dirías que los datos están
desbalanceados? (c) Mirando los promedios por grupo: ¿qué variable muestra la diferencia más grande
entre diabéticas y no diabéticas?

**Respuesta:**


## Parte 1 — Árbol con impureza de Gini  *(presentación B1–B2, B5)*

Dividimos en entrenamiento/prueba de forma estratificada y elegimos `max_depth` con validación cruzada.


In [ ]:
X = diabetes.drop(columns="diabetico")
y = diabetes["diabetico"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

malla_prof = {"max_depth": list(range(2, 11))}
arbol_gini = ...     # ✏️ GridSearchCV(DecisionTreeClassifier(criterion="gini", random_state=0), malla_prof, cv=kf, scoring="accuracy").fit(Xtr, ytr)
print("Gini — mejor profundidad:", arbol_gini.best_params_["max_depth"],
      "   exactitud de validación:", round(arbol_gini.best_score_, 3))
print("Exactitud en prueba:", round(arbol_gini.score(Xte, yte), 3))
print(export_text(arbol_gini.best_estimator_, feature_names=list(X.columns), decimals=1, max_depth=2))


✏️ **Pregunta 1.1.** (a) ¿Qué profundidad eligió la validación cruzada para el árbol con Gini? (b) Lee
las primeras líneas del árbol impreso: ¿qué variable y qué valor usa en la raíz? ¿Coincide con tu
respuesta de la Pregunta 0.1(c)?

**Respuesta:**


## Parte 2 — Árbol con entropía: ¿cambia mucho?  *(presentación B2–B3, B6)*

Repetimos el mismo procedimiento con `criterion="entropy"` y comparamos.


In [ ]:
arbol_entropia = ...     # ✏️ GridSearchCV(DecisionTreeClassifier(criterion="entropy", random_state=0), malla_prof, cv=kf, scoring="accuracy").fit(Xtr, ytr)
print("Entropía — mejor profundidad:", arbol_entropia.best_params_["max_depth"],
      "   exactitud de validación:", round(arbol_entropia.best_score_, 3))
print("Exactitud en prueba:", round(arbol_entropia.score(Xte, yte), 3))

comparacion = pd.DataFrame({
    "criterio": ["gini", "entropia"],
    "profundidad": [arbol_gini.best_params_["max_depth"], arbol_entropia.best_params_["max_depth"]],
    "exactitud_cv": [round(arbol_gini.best_score_, 3), round(arbol_entropia.best_score_, 3)],
    "exactitud_prueba": [round(arbol_gini.score(Xte, yte), 3), round(arbol_entropia.score(Xte, yte), 3)],
})
comparacion


✏️ **Pregunta 2.1.** ¿Cambia mucho el resultado entre Gini y entropía (profundidad elegida, exactitud
de validación y de prueba)? A partir de lo que viste en la presentación sobre cómo se comparan ambas
medidas de impureza en una misma gráfica, ¿esperabas una diferencia grande o pequeña?

**Respuesta:**


## Parte 3 — Evaluación del mejor árbol  *(presentación C1–C7)*

Nos quedamos con el árbol de entropía (empató o superó ligeramente al de Gini) y lo evaluamos **una
sola vez** sobre el conjunto de prueba: matriz de confusión, precisión, recall, F1, curva ROC y AUC.


In [ ]:
mejor_arbol = arbol_entropia.best_estimator_
pred = mejor_arbol.predict(Xte)
print(classification_report(yte, pred, target_names=["no diabetico", "diabetico"]))
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=["no diabetico", "diabetico"],
                                         cmap="Blues", colorbar=False)
plt.title("Mejor árbol (entropía, profundidad 4)"); plt.show()


In [ ]:
proba_arbol = mejor_arbol.predict_proba(Xte)[:, 1]
fpr, tpr, umbrales = ...     # ✏️ roc_curve(yte, proba_arbol)
auc_arbol = roc_auc_score(yte, proba_arbol)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(fpr, tpr, color="#065A82", label=f"árbol (AUC = {auc_arbol:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="azar (AUC = 0.5)")
ax.set_xlabel("tasa de falsos positivos"); ax.set_ylabel("tasa de verdaderos positivos")
ax.set_title("Curva ROC — árbol de decisión"); ax.legend()
plt.show()
print("AUC del árbol:", round(auc_arbol, 3))


✏️ **Pregunta 3.1.** (a) Según la matriz de confusión, ¿cuántos falsos negativos (diabéticas no
detectadas) comete el árbol? (b) ¿Qué AUC obtuvo? ¿Es un valor cercano a 1 (clasificador casi
perfecto), cercano a 0.5 (equivalente al azar), o intermedio?

**Respuesta:**


## Parte 4 — Árbol vs. Naive Bayes gaussiano  *(presentación A5, B7)*

Entrenamos `GaussianNB` con las mismas 6 variables y comparamos su AUC contra el del árbol.


In [ ]:
gnb = ...     # ✏️ GaussianNB().fit(Xtr, ytr)
auc_gnb = roc_auc_score(yte, gnb.predict_proba(Xte)[:, 1])
fpr_g, tpr_g, _ = roc_curve(yte, gnb.predict_proba(Xte)[:, 1])

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(fpr, tpr, color="#065A82", label=f"árbol (AUC = {auc_arbol:.3f})")
ax.plot(fpr_g, tpr_g, color="#E07A1F", label=f"GaussianNB (AUC = {auc_gnb:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="azar")
ax.set_xlabel("tasa de falsos positivos"); ax.set_ylabel("tasa de verdaderos positivos")
ax.set_title("Árbol vs. Naive Bayes gaussiano"); ax.legend()
plt.show()
print(f"Exactitud  — árbol: {mejor_arbol.score(Xte, yte):.3f}   GaussianNB: {gnb.score(Xte, yte):.3f}")
print(f"AUC        — árbol: {auc_arbol:.3f}   GaussianNB: {auc_gnb:.3f}")


✏️ **Pregunta 4.1.** (a) ¿Cuál de los dos modelos tiene mejor AUC? (b) Recordando el supuesto "naive"
de independencia condicional de Naive Bayes y la forma en que un árbol construye sus cortes: ¿por qué
tendría sentido que, en variables continuas con relaciones suaves como estas, Naive Bayes gaussiano
compita bien contra un árbol poco profundo? Escribe tu conclusión: ¿cuál recomendarías y por qué.

**Respuesta:**
